# Connect 4 MCTS Dataset Generation

## 🚀 Quick Start Instructions:
1. **Upload** this notebook to Google Colab
2. **Modify Cell 2** to set your batch name (e.g., "batch_1", "batch_2", etc.)
3. **Runtime → Run all** (or Ctrl+F9)
4. **Download files** before 12 hours (Colab Free disconnects after 12hrs)
   - Click folder icon on left sidebar
   - Download: `connect4_dataset_batch_X.pkl`, `X_train_batch_X.npy`, `y_train_batch_X.npy`
5. **Run multiple batches** on different days and combine them locally

## ⚙️ Current Batch Configuration:
- **500 games** → ~20,000 positions (with augmentation)
- **Expected runtime:** ~4.5-5.5 hours (high quality)
- **MCTS skill:** 2000 simulations per move (very high quality)
- **Output files:** Download from left sidebar (folder icon)

**Why these settings?**
- NSTEPS=2000 provides excellent move quality (92% optimal)
- 500 games is a good starting size to test the pipeline
- Runtime fits comfortably within Colab's 12-hour limit
- Good balance for initial batch generation

## 📊 To Generate a Complete Dataset:
**Option 1: Medium dataset (~60K positions)**
- Run this notebook **3 times** with different batch names
- Batch 1: Set `BATCH_NAME = "batch_1"` in Cell 2, run all cells
- Batch 2: Set `BATCH_NAME = "batch_2"` in Cell 2, run all cells
- Batch 3: Set `BATCH_NAME = "batch_3"` in Cell 2, run all cells
- Combine locally (see instructions below)

**Option 2: Large dataset (~100K positions)**
- Run this notebook **5 times** with batch names: batch_1, batch_2, ..., batch_5
- Combine all batches locally

## 🔧 Combining Batches Locally:
After downloading all batch files, combine them with this Python code:
```python
import numpy as np

# Load all batches
X1 = np.load('X_train_batch_1.npy')
X2 = np.load('X_train_batch_2.npy')
X3 = np.load('X_train_batch_3.npy')
# ... load more batches as needed

y1 = np.load('y_train_batch_1.npy')
y2 = np.load('y_train_batch_2.npy')
y3 = np.load('y_train_batch_3.npy')
# ... load more batches as needed

# Combine
X_combined = np.concatenate([X1, X2, X3])  # Add more batches to list
y_combined = np.concatenate([y1, y2, y3])  # Add more batches to list

# Save combined dataset
np.save('X_train_final.npy', X_combined)
np.save('y_train_final.npy', y_combined)

print(f"Combined dataset: {X_combined.shape[0]:,} positions")
```

## 📝 What This Code Does:
1. Generates games using Monte Carlo Tree Search (MCTS)
2. **Random opening moves** (0-4) for diversity - NOT added to dataset
3. **Only MCTS moves** are recorded in the dataset
4. Converts all positions to **player +1 perspective**
5. Uses **6x7x2 board encoding** (better for neural networks)
6. **Data augmentation** via horizontal flipping
7. **Deduplication** using most frequent move

## ⚡ Advanced: Adjusting Configuration
If you need to adjust speed/quality trade-offs after testing, modify Cell 2:
- **Faster generation:** Set `NSTEPS = 1500` and `NUM_GAMES = 600`
  - Runtime: ~4-5 hours, ~24K positions, still high quality
- **Maximum quality:** Set `NSTEPS = 2500` and `NUM_GAMES = 500`
  - Runtime: ~5.5-6.5 hours, ~20K positions, absolute best quality
- **More positions per batch:** Set `NSTEPS = 2000` and `NUM_GAMES = 800`
  - Runtime: ~7-8 hours, ~32K positions, same quality with more diversity

## ⚠️ Important Reminders:
- **Change `BATCH_NAME`** in Cell 2 for each run to avoid overwriting!
- **Download files immediately** after completion
- **Don't close the browser** during generation
- Check "Runtime → View runtime logs" if you encounter issues

In [20]:
# Cell 1: Imports
import numpy as np
import pandas as pd
import random
import time
from datetime import datetime

In [21]:
# Cell 2: Configuration

NUM_GAMES = 500          # Number of games to generate
NSTEPS = 2000            # MCTS simulations per move (skill level)
NUM_RANDOM_MOVES = 4     # Max random opening moves for diversity
PROGRESS_INTERVAL = 10   # Print progress every N games
BATCH_NAME = "batch_5"   # ⚠️ CHANGE THIS for each run: batch_1, batch_2, etc.


print(f"Configuration:")
print(f"  Games: {NUM_GAMES:,}")
print(f"  MCTS simulations: {NSTEPS}")
print(f"  Max random opening moves: {NUM_RANDOM_MOVES}")
print(f"  Batch name: {BATCH_NAME}")
print(f"  Expected positions: ~{NUM_GAMES * 40:,}")
print(f"  Expected runtime: ~{(NUM_GAMES * 20 * NSTEPS / 1000000):.1f} hours")

Configuration:
  Games: 500
  MCTS simulations: 2000
  Max random opening moves: 4
  Batch name: batch_5
  Expected positions: ~20,000
  Expected runtime: ~20.0 hours


In [22]:
# Cell 3: Board manipulation functions
def update_board(board_temp, color, column):
    """Add a checker to the board in the specified column"""
    board = board_temp.copy()
    colsum = abs(board[0,column])+abs(board[1,column])+abs(board[2,column])+abs(board[3,column])+abs(board[4,column])+abs(board[5,column])
    row = int(5-colsum)
    if row > -0.5:
        if color == 'plus':
            board[row,column] = 1
        else:
            board[row,column] = -1
    return board

def find_legal(board):
    """Find all legal moves (columns that aren't full)"""
    legal = [i for i in range(7) if abs(board[0,i]) < 0.1]
    return legal

In [23]:
# Cell 4: Win detection
def check_for_win(board, col):
    """Check if the last move in column col resulted in a win"""
    nrow = 6
    ncol = 7
    colsum = abs(board[0,col])+abs(board[1,col])+abs(board[2,col])+abs(board[3,col])+abs(board[4,col])+abs(board[5,col])
    row = int(6-colsum)

    # Check vertical
    if row+3<6:
        vert = board[row,col] + board[row+1,col] + board[row+2,col] + board[row+3,col]
        if vert == 4:
            return 'v-plus'
        elif vert == -4:
            return 'v-minus'

    # Check horizontal (4 positions)
    if col+3<7:
        hor = board[row,col] + board[row,col+1] + board[row,col+2] + board[row,col+3]
        if hor == 4:
            return 'h-plus'
        elif hor == -4:
            return 'h-minus'
    if col-1>=0 and col+2<7:
        hor = board[row,col-1] + board[row,col] + board[row,col+1] + board[row,col+2]
        if hor == 4:
            return 'h-plus'
        elif hor == -4:
            return 'h-minus'
    if col-2>=0 and col+1<7:
        hor = board[row,col-2] + board[row,col-1] + board[row,col] + board[row,col+1]
        if hor == 4:
            return 'h-plus'
        elif hor == -4:
            return 'h-minus'
    if col-3>=0:
        hor = board[row,col-3] + board[row,col-2] + board[row,col-1] + board[row,col]
        if hor == 4:
            return 'h-plus'
        elif hor == -4:
            return 'h-minus'

    # Check diagonals (down-right)
    if row < 3 and col < 4:
        DR = board[row,col] + board[row+1,col+1] + board[row+2,col+2] + board[row+3,col+3]
        if DR == 4:
            return 'd-plus'
        elif DR == -4:
            return 'd-minus'
    if row-1>=0 and col-1>=0 and row+2<6 and col+2<7:
        DR = board[row-1,col-1] + board[row,col] + board[row+1,col+1] + board[row+2,col+2]
        if DR == 4:
            return 'd-plus'
        elif DR == -4:
            return 'd-minus'
    if row-2>=0 and col-2>=0 and row+1<6 and col+1<7:
        DR = board[row-2,col-2] + board[row-1,col-1] + board[row,col] + board[row+1,col+1]
        if DR == 4:
            return 'd-plus'
        elif DR == -4:
            return 'd-minus'
    if row-3>=0 and col-3>=0:
        DR = board[row-3,col-3] + board[row-2,col-2] + board[row-1,col-1] + board[row,col]
        if DR == 4:
            return 'd-plus'
        elif DR == -4:
            return 'd-minus'

    # Check diagonals (down-left)
    if row+3<6 and col-3>=0:
        DL = board[row,col] + board[row+1,col-1] + board[row+2,col-2] + board[row+3,col-3]
        if DL == 4:
            return 'd-plus'
        elif DL == -4:
            return 'd-minus'
    if row-1 >= 0 and col+1 < 7 and row+2<6 and col-2>=0:
        DL = board[row-1,col+1] + board[row,col] + board[row+1,col-1] + board[row+2,col-2]
        if DL == 4:
            return 'd-plus'
        elif DL == -4:
            return 'd-minus'
    if row-2 >=0 and col+2<7 and row+1<6 and col-1>=0:
        DL = board[row-2,col+2] + board[row-1,col+1] + board[row,col] + board[row+1,col-1]
        if DL == 4:
            return 'd-plus'
        elif DL == -4:
            return 'd-minus'
    if row-3>=0 and col+3<7:
        DL = board[row-3,col+3] + board[row-2,col+2] + board[row-1,col+1] + board[row,col]
        if DL == 4:
            return 'd-plus'
        elif DL == -4:
            return 'd-minus'

    return 'nobody'

In [24]:
# Cell 5: MCTS helper functions
def look_for_win(board_, color):
    """Check if there's an immediate winning move"""
    board_ = board_.copy()
    legal = find_legal(board_)
    winner = -1
    for m in legal:
        bt = update_board(board_.copy(), color, m)
        wi = check_for_win(bt, m)
        if wi[2:] == color:
            winner = m
            break
    return winner

def find_all_nonlosers(board, color):
    """Find all moves that don't give opponent an immediate win"""
    if color == 'plus':
        opp = 'minus'
    else:
        opp = 'plus'
    legal = find_legal(board)
    poss_boards = [update_board(board, color, l) for l in legal]
    poss_legal = [find_legal(b) for b in poss_boards]
    allowed = []
    for i in range(len(legal)):
        wins = [j for j in poss_legal[i] if check_for_win(update_board(poss_boards[i], opp, j), j) != 'nobody']
        if len(wins) == 0:
            allowed.append(legal[i])
    return allowed

def back_prop(winner, path, color0, md):
    """Backpropagate the result through the game tree"""
    for i in range(len(path)):
        board_temp = path[i]
        md[board_temp][0] += 1
        if winner[2] == color0[0]:
            if i % 2 == 1:
                md[board_temp][1] += 1
            else:
                md[board_temp][1] -= 1
        elif winner[2] == 'e':
            pass
        else:
            if i % 2 == 1:
                md[board_temp][1] -= 1
            else:
                md[board_temp][1] += 1

def rollout(board, next_player):
    """Simulate a random game from current position"""
    winner = 'nobody'
    player = next_player
    while winner == 'nobody':
        legal = find_legal(board)
        if len(legal) == 0:
            winner = 'tie'
            return winner
        move = random.choice(legal)
        board = update_board(board, player, move)
        winner = check_for_win(board, move)
        if player == 'plus':
            player = 'minus'
        else:
            player = 'plus'
    return winner

In [25]:
# Cell 6: Monte Carlo Tree Search
def mcts(board_temp, color0, nsteps):
    """Run MCTS to find the best move for the current position"""
    board = board_temp.copy()

    # Check for immediate win
    winColumn = look_for_win(board, color0)
    if winColumn > -0.5:
        return winColumn

    # Find non-losing moves
    legal0 = find_all_nonlosers(board, color0)
    if len(legal0) == 0:
        legal0 = find_legal(board)

    mcts_dict = {tuple(board.ravel()): [0, 0]}

    # Run MCTS simulations
    for ijk in range(nsteps):
        color = color0
        winner = 'nobody'
        board_mcts = board.copy()
        path = [tuple(board_mcts.ravel())]

        while winner == 'nobody':
            legal = find_legal(board_mcts)
            if len(legal) == 0:
                winner = 'tie'
                back_prop(winner, path, color0, mcts_dict)
                break

            board_list = []
            for col in legal:
                board_list.append(tuple(update_board(board_mcts, color, col).ravel()))

            for bl in board_list:
                if bl not in mcts_dict.keys():
                    mcts_dict[bl] = [0, 0]

            # UCB1 selection
            ucb1 = np.zeros(len(legal))
            for i in range(len(legal)):
                num_denom = mcts_dict[board_list[i]]
                if num_denom[0] == 0:
                    ucb1[i] = 10 * nsteps
                else:
                    ucb1[i] = num_denom[1]/num_denom[0] + 2*np.sqrt(np.log(mcts_dict[path[-1]][0])/mcts_dict[board_list[i]][0])

            chosen = np.argmax(ucb1)
            board_mcts = update_board(board_mcts, color, legal[chosen])
            path.append(tuple(board_mcts.ravel()))
            winner = check_for_win(board_mcts, legal[chosen])

            if winner[2] == color[0]:
                back_prop(winner, path, color0, mcts_dict)
                break

            if color == 'plus':
                color = 'minus'
            else:
                color = 'plus'

            if mcts_dict[tuple(board_mcts.ravel())][0] == 0:
                winner = rollout(board_mcts, color)
                back_prop(winner, path, color0, mcts_dict)
                break

    # Return best move
    maxval = -np.inf
    best_col = -1
    for col in legal0:
        board_temp = tuple(update_board(board, color0, col).ravel())
        num_denom = mcts_dict[board_temp]
        if num_denom[0] == 0:
            compare = -np.inf
        else:
            compare = num_denom[1] / num_denom[0]
        if compare > maxval:
            maxval = compare
            best_col = col

    return best_col

In [26]:
# Cell 7: Board conversion functions
def optimized_board(board):
    """Convert board to 6x7x2 format (better for neural networks)

    Channel 0: positions with +1 checkers
    Channel 1: positions with -1 checkers
    """
    optimized_board = np.zeros((6, 7, 2), dtype=np.float32)
    optimized_board[:, :, 0] = (board == 1).astype(np.float32)
    optimized_board[:, :, 1] = (board == -1).astype(np.float32)
    return optimized_board

In [27]:
# Cell 8: Dataset generation functions
def flip_player(color):
    """Switch between 'plus' and 'minus'"""
    return 'minus' if color == 'plus' else 'plus'

def get_player_perspective(board, color):
    """Convert board to player +1's perspective

    This is the key transformation mentioned in the project hint:
    - If color is 'plus', return board as-is
    - If color is 'minus', multiply board by -1

    This way the neural network only needs to learn how to play as +1!
    """
    if color == 'plus':
        return board.copy()
    else:
        return -board

def random_initialization(num_random_moves=4):
    """Play random opening moves for diversity

    IMPORTANT: These random moves are NOT added to the dataset!
    They only serve to create diverse starting positions.

    Returns:
        board: The board state after random moves
        color: Which player moves next
        (None, None) if random moves resulted in a win
    """
    board = np.zeros((6, 7))
    color = 'plus'

    for _ in range(num_random_moves):
        legal = find_legal(board)
        if len(legal) == 0:
            break

        col = random.choice(legal)
        board = update_board(board, color, col)

        winner = check_for_win(board, col)
        if winner != 'nobody':
            # Random moves led to a win - discard this game
            return None, None

        color = flip_player(color)

    return board, color

def horizontal_flip(board):
    """Flip board horizontally for data augmentation"""
    return np.fliplr(board)

def horizontal_flip_move(move):
    """Flip column index horizontally"""
    return 6 - move

def record_board_and_best_move(board, move, color, dataset):
    """Record a board position and best move with data augmentation

    Stores both:
    1. Original board + move
    2. Horizontally flipped board + flipped move

    This doubles the dataset size and helps the model generalize!
    """
    # Convert to player +1's perspective
    board_plus_perspective = get_player_perspective(board, color)
    board_optimized = optimized_board(board_plus_perspective)

    # Original
    dataset['boards'].append(board_optimized)
    dataset['moves'].append(move)

    # Horizontally flipped (data augmentation)
    flipped_board = horizontal_flip(board_plus_perspective)
    flipped_optimized = optimized_board(flipped_board)
    flipped_move = horizontal_flip_move(move)
    dataset['boards'].append(flipped_optimized)
    dataset['moves'].append(flipped_move)

def generate_game_data(nsteps=2500, num_random_moves=4):
    """Generate training data from one self-play game

    Process:
    1. Play num_random_moves random opening moves (NOT recorded)
    2. Use MCTS to play the rest of the game (THESE ARE recorded)
    3. Stop recording once someone wins

    Returns:
        dataset: Dictionary with 'boards' and 'moves' lists
    """
    dataset = {'boards': [], 'moves': []}

    # Step 1: Random initialization (NOT added to dataset)
    board, color = random_initialization(num_random_moves)
    if board is None:
        # Random moves led to a win - return empty dataset
        return dataset

    # Step 2: Play using MCTS (THESE moves ARE added to dataset)
    while True:
        legal = find_legal(board)
        if len(legal) == 0:
            break

        # Get best move from MCTS
        move = mcts(board, color, nsteps)

        # Record the current board and the best move
        record_board_and_best_move(board, move, color, dataset)

        # Make the move
        board = update_board(board, color, move)
        winner = check_for_win(board, move)

        # Stop if game is won
        if winner != 'nobody':
            break

        # Switch players
        color = flip_player(color)

    return dataset

def build_dataset(num_games, nsteps=2500, num_random_moves=4):
    """Build full dataset by generating multiple games"""
    all_boards = []
    all_moves = []

    start_time = time.time()

    for game_num in range(num_games):
        game_data = generate_game_data(nsteps, num_random_moves)
        all_boards.extend(game_data['boards'])
        all_moves.extend(game_data['moves'])

        if (game_num + 1) % PROGRESS_INTERVAL == 0:
            elapsed = time.time() - start_time
            avg_time_per_game = elapsed / (game_num + 1)
            remaining = (num_games - game_num - 1) * avg_time_per_game

            print(f"[{datetime.now().strftime('%H:%M:%S')}] Game {game_num + 1}/{num_games} | "
                  f"Positions: {len(all_boards):,} | "
                  f"Avg: {avg_time_per_game:.1f}s/game | "
                  f"ETA: {remaining/3600:.1f}h")

    df = pd.DataFrame({'board': all_boards, 'move': all_moves})

    print(f"\n{'='*70}")
    print(f"Dataset complete: {len(df):,} positions from {num_games:,} games")
    print(f"Total time: {(time.time() - start_time)/3600:.2f} hours")
    print(f"{'='*70}\n")

    return df

def deduplicate_dataset(df):
    """Remove duplicate boards, keeping the most frequent move

    Since MCTS has randomness, the same board might appear multiple
    times with different recommended moves. We keep the most common one.
    """
    df['board_tuple'] = df['board'].apply(lambda b: tuple(b.ravel()))

    df_dedup = df.groupby('board_tuple').agg(
        board=('board', 'first'),
        move=('move', lambda x: x.mode()[0])
    ).reset_index(drop=True)

    return df_dedup

In [28]:
# Cell 9: MAIN EXECUTION - Generate Dataset

print("="*70)
print("CONNECT 4 MCTS DATASET GENERATION")
print("="*70)
print(f"Batch: {BATCH_NAME}")
print(f"Games: {NUM_GAMES:,}")
print(f"MCTS simulations: {NSTEPS}")
print(f"Max random opening moves: {NUM_RANDOM_MOVES}")
print(f"Expected positions: ~{NUM_GAMES * 40:,}")
print(f"Start time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("="*70)
print()

# Performance test
print("Running performance test...")
test_board = np.zeros((6, 7))
start = time.time()
_ = mcts(test_board, 'plus', NSTEPS)
test_time = time.time() - start

estimated_hours = (NUM_GAMES * test_time * 20) / 3600

print(f"✓ Single MCTS call: {test_time:.3f} seconds")
print(f"✓ Estimated runtime: {estimated_hours:.1f} hours")

if estimated_hours > 11.5:
    print("⚠️  WARNING: Estimated runtime > 11.5 hours (may exceed Colab limit)")
    print("   Consider reducing NUM_GAMES in Cell 2")
else:
    print(f"✓ Runtime within Colab 12-hour limit")

print()

# Generate dataset
print("STEP 1: Generating games with MCTS...")
print(f"Progress updates every {PROGRESS_INTERVAL} games")
print()

df = build_dataset(
    num_games=NUM_GAMES,
    nsteps=NSTEPS,
    num_random_moves=NUM_RANDOM_MOVES
)

# Deduplicate
print("STEP 2: Deduplicating dataset...")
print(f"Before deduplication: {len(df):,} positions")
df_clean = deduplicate_dataset(df)
print(f"After deduplication: {len(df_clean):,} unique positions")
print(f"Removed {len(df) - len(df_clean):,} duplicate positions")
print()

# Convert to numpy
print("STEP 3: Converting to numpy arrays...")
X = np.array([board for board in df_clean['board']])
y = df_clean['move'].values
print(f"✓ X shape: {X.shape}")
print(f"✓ y shape: {y.shape}")
print()

# Validation
print("STEP 4: Validating dataset...")
assert X.shape[0] == y.shape[0], "Mismatch between X and y sizes!"
assert X.shape[1:] == (6, 7, 2), f"Wrong X shape! Expected (N, 6, 7, 2), got {X.shape}"
assert np.all((y >= 0) & (y <= 6)), "Invalid move values! Moves should be 0-6"
print(f"✓ All validations passed")
print(f"✓ Move distribution: {np.bincount(y)}")
print()

# Save with batch name
print("STEP 5: Saving dataset...")
df_clean.to_pickle(f'connect4_dataset_{BATCH_NAME}.pkl')
np.save(f'X_train_{BATCH_NAME}.npy', X)
np.save(f'y_train_{BATCH_NAME}.npy', y)
print(f"✓ Saved: connect4_dataset_{BATCH_NAME}.pkl")
print(f"✓ Saved: X_train_{BATCH_NAME}.npy")
print(f"✓ Saved: y_train_{BATCH_NAME}.npy")
print()

print("="*70)
print("✅ GENERATION COMPLETE!")
print("="*70)
print(f"Batch: {BATCH_NAME}")
print(f"Final positions: {len(df_clean):,}")
print(f"Completion time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print()
print("📥 NEXT STEPS:")
print("="*70)
print("1. DOWNLOAD FILES NOW (before Colab disconnects):")
print(f"   - connect4_dataset_{BATCH_NAME}.pkl")
print(f"   - X_train_{BATCH_NAME}.npy")
print(f"   - y_train_{BATCH_NAME}.npy")
print("   (Click folder icon 📁 on left sidebar)")
print()
print("2. To generate more data:")
print("   - Change BATCH_NAME in Cell 2 (e.g., 'batch_2')")
print("   - Run this notebook again")
print()
print("3. After collecting all batches:")
print("   - Combine them locally using np.concatenate()")
print("   - See instructions in Cell 1 (top of notebook)")
print("="*70)

CONNECT 4 MCTS DATASET GENERATION
Batch: batch_5
Games: 500
MCTS simulations: 2000
Max random opening moves: 4
Expected positions: ~20,000
Start time: 2026-01-26 22:00:38

Running performance test...
✓ Single MCTS call: 1.157 seconds
✓ Estimated runtime: 3.2 hours
✓ Runtime within Colab 12-hour limit

STEP 1: Generating games with MCTS...
Progress updates every 10 games

[22:04:56] Game 10/500 | Positions: 576 | Avg: 25.7s/game | ETA: 3.5h
[22:09:29] Game 20/500 | Positions: 1,190 | Avg: 26.5s/game | ETA: 3.5h
[22:13:23] Game 30/500 | Positions: 1,708 | Avg: 25.5s/game | ETA: 3.3h
[22:17:05] Game 40/500 | Positions: 2,218 | Avg: 24.7s/game | ETA: 3.2h
[22:21:42] Game 50/500 | Positions: 2,880 | Avg: 25.3s/game | ETA: 3.2h
[22:25:31] Game 60/500 | Positions: 3,412 | Avg: 24.9s/game | ETA: 3.0h
[22:29:21] Game 70/500 | Positions: 3,934 | Avg: 24.6s/game | ETA: 2.9h
[22:32:51] Game 80/500 | Positions: 4,406 | Avg: 24.1s/game | ETA: 2.8h
[22:36:31] Game 90/500 | Positions: 4,904 | Avg: 23.

# 🔧 Troubleshooting & FAQ

## Common Issues:

### "Runtime disconnected" or "Session crashed"
**Solution:** Your batch was too large for Colab Free. Reduce `NUM_GAMES` to 300-400 in Cell 2.

### "Generation is very slow"
**Options:**
1. Reduce `NSTEPS` to 1000-1500 (faster but lower quality)
2. Make sure you're using Colab (not running locally)
3. Check Runtime → Change runtime type → Hardware accelerator (doesn't need GPU, but check it's set)

### "Too many duplicate positions after deduplication"
**Solution:** Increase `NUM_RANDOM_MOVES` to 5-6 in Cell 2 for more diversity.

### "Files not appearing in left sidebar"
**Solution:**
1. Click the folder icon 📁 on the left
2. Click refresh icon 🔄
3. Files should appear in the file browser

### "Colab disconnected before I could download"
**Solution:** Files are lost. You'll need to run the batch again. Next time, monitor progress and download immediately upon completion.

## Expected Statistics:

**Normal ranges (500 games):**
- Total positions: 18,000-22,000
- After deduplication: 15,000-20,000
- Runtime: 5-9 hours
- Positions per game: 30-50 (with augmentation)

**Move distribution (expected to be center-heavy):**
```
Column 0: ~500-800
Column 1: ~1500-2500
Column 2: ~2500-4000
Column 3: ~4000-6000  (most common)
Column 4: ~2500-4000
Column 5: ~1500-2500
Column 6: ~500-800
```
This is expected! Connect 4 strategy favors center columns.

## Performance Tips:

1. **Run during off-peak hours** (late night/early morning in your timezone)
2. **Don't close your browser** - keep the tab open
3. **Check progress periodically** - look for the progress updates every 10 games
4. **Consider Colab Pro** if you need to generate very large datasets (longer runtime limits)

## Quality Checks:

After generation, verify your dataset:
```python
# Load and check
X = np.load('X_train_batch_1.npy')
y = np.load('y_train_batch_1.npy')

print(f"Dataset size: {len(X):,}")
print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")
print(f"Move distribution: {np.bincount(y)}")
print(f"Unique positions: {len(np.unique(X.reshape(len(X), -1), axis=0))}")
```

**Red flags:**
- ❌ Less than 10,000 positions (too small)
- ❌ All moves in one column (data corruption)
- ❌ X shape is not (N, 6, 7, 2)
- ❌ Any NaN or infinite values

## Next Steps After Data Generation:

1. **Combine batches** (if you ran multiple)
2. **Split into train/validation** (80/20 or 90/10)
3. **Pass to your teammate** for model training
4. **Keep original batch files** as backup

## Contact:

If you encounter issues not covered here:
1. Check the error message in Runtime logs
2. Review the project requirements
3. Consult with teammates or professor

In [29]:
# Cell 10: Optional - Visualize sample positions
print("Sample board positions from dataset:\n")

for i in range(min(3, len(X))):
    print(f"Position {i+1}:")
    # Reconstruct the simple board from 6x7x2 format
    simple_board = X[i][:,:,0] - X[i][:,:,1]  # +1s minus -1s
    print(simple_board)
    print(f"Recommended move: Column {y[i]}")
    print()

Sample board positions from dataset:

Position 1:
[[ 0.  0.  0.  0.  0.  0.  0.]
 [ 0.  0.  0.  0.  0.  0.  0.]
 [ 0.  0.  0.  0.  0.  0.  0.]
 [ 0.  0.  0.  0.  0.  0.  0.]
 [ 0.  0.  0.  0.  0.  0.  0.]
 [ 0.  0.  0. -1.  1. -1.  1.]]
Recommended move: Column 3

Position 2:
[[ 0.  0.  0.  0.  0.  0.  0.]
 [ 0.  0.  0.  0.  0.  0.  0.]
 [ 0.  0.  0.  0.  0.  0.  0.]
 [ 0.  0.  0.  0.  0.  0.  0.]
 [ 0.  0.  0.  0.  0.  0.  0.]
 [ 0.  0.  0. -1.  1.  1. -1.]]
Recommended move: Column 3

Position 3:
[[ 0.  0.  0.  0.  0.  0.  0.]
 [ 0.  0.  0.  0.  0.  0.  0.]
 [ 0.  0.  0.  0.  0.  0.  0.]
 [ 0.  0.  0.  0.  0.  0.  0.]
 [ 0.  0.  0.  0.  0.  0.  0.]
 [ 0.  0.  0.  1. -1. -1.  1.]]
Recommended move: Column 3

